# CODTECH ML Internship — Task 3
## Image Classification Model (CNN)
**Objective:** Build a Convolutional Neural Network (CNN) for image classification using TensorFlow/Keras on the CIFAR-10 dataset.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')

print('TensorFlow version:', tf.__version__)
print('GPU available:', len(tf.config.list_physical_devices("GPU")) > 0)

## 1. Load & Explore CIFAR-10 Dataset

In [ ]:
# CIFAR-10: 60,000 color images (32x32) across 10 classes
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Class names for visualization
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f'Training set shape  : {X_train.shape}')
print(f'Test set shape      : {X_test.shape}')
print(f'Labels shape        : {y_train.shape}')
print(f'Pixel value range   : [{X_train.min()}, {X_train.max()}]')

In [ ]:
# Visualize sample images from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for idx, ax in enumerate(axes.flat):
    # Find first occurrence of each class
    class_idx = np.where(y_train.flatten() == idx)[0][0]
    ax.imshow(X_train[class_idx])
    ax.set_title(CLASS_NAMES[idx], fontsize=12)
    ax.axis('off')
plt.suptitle('CIFAR-10 Sample Images (One Per Class)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('cifar10_samples.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. Preprocessing

In [ ]:
# Normalize pixel values from [0, 255] to [0.0, 1.0]
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32')  / 255.0

# One-hot encode labels: e.g. class 3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat  = to_categorical(y_test,  num_classes=10)

print('Preprocessing complete.')
print(f'X_train range: [{X_train.min():.1f}, {X_train.max():.1f}]')
print(f'y_train_cat shape: {y_train_cat.shape}')

## 3. Data Augmentation

In [ ]:
# Augment training images to improve generalization
datagen = ImageDataGenerator(
    rotation_range=15,       # Randomly rotate images up to 15 degrees
    width_shift_range=0.1,   # Randomly shift images horizontally
    height_shift_range=0.1,  # Randomly shift images vertically
    horizontal_flip=True,    # Randomly flip images left-to-right
    zoom_range=0.1           # Randomly zoom into images
)

datagen.fit(X_train)
print('Data augmentation configured.')

## 4. Build the CNN Architecture

In [ ]:
def build_cnn(input_shape=(32, 32, 3), num_classes=10):
    """
    CNN Architecture:
    Block 1: Conv(32) → Conv(32) → MaxPool → Dropout
    Block 2: Conv(64) → Conv(64) → MaxPool → Dropout
    Block 3: Conv(128) → Conv(128) → MaxPool → Dropout
    Head   : Flatten → Dense(256) → Dropout → Softmax
    """
    model = models.Sequential([
        # ── Input Layer ──────────────────────────────────
        layers.Input(shape=input_shape),

        # ── Convolutional Block 1 ─────────────────────────
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # ── Convolutional Block 2 ─────────────────────────
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # ── Convolutional Block 3 ─────────────────────────
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # ── Fully Connected Head ──────────────────────────
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),

        # ── Output Layer ──────────────────────────────────
        layers.Dense(num_classes, activation='softmax')  # 10-class probability
    ])
    return model

model = build_cnn()

# Compile the model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Train the Model

In [ ]:
# Callbacks to prevent overfitting and adjust learning rate
callbacks = [
    EarlyStopping(
        monitor='val_accuracy', patience=10,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    )
]

BATCH_SIZE = 64
EPOCHS     = 50  # Early stopping will kick in before this

# Train using augmented data
history = model.fit(
    datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE),
    steps_per_epoch=len(X_train) // BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(X_test, y_test_cat),
    callbacks=callbacks,
    verbose=1
)

## 6. Training History Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy plot
axes[0].plot(history.history['accuracy'],     label='Train Accuracy', color='royalblue')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy',   color='darkorange')
axes[0].set_title('Model Accuracy Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Loss plot
axes[1].plot(history.history['loss'],     label='Train Loss', color='royalblue')
axes[1].plot(history.history['val_loss'], label='Val Loss',   color='darkorange')
axes[1].set_title('Model Loss Over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('CNN Training History — CIFAR-10', fontsize=14)
plt.tight_layout()
plt.savefig('training_history.png', dpi=100, bbox_inches='tight')
plt.show()

## 7. Performance Evaluation on Test Set

In [ ]:
# Evaluate model on the test set
test_loss, test_acc = model.evaluate(X_test, y_test_cat, verbose=0)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc * 100:.2f}%')

# Generate class-wise predictions
y_pred_probs = model.predict(X_test, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = y_test.flatten()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix — CIFAR-10 CNN', fontsize=14)
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.tight_layout()
plt.savefig('cnn_confusion_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Sample Predictions Visualization

In [ ]:
# Show 15 test images with true vs predicted labels
fig, axes = plt.subplots(3, 5, figsize=(15, 9))

for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i])
    true_label = CLASS_NAMES[y_true[i]]
    pred_label = CLASS_NAMES[y_pred[i]]
    conf       = y_pred_probs[i, y_pred[i]] * 100
    color      = 'green' if true_label == pred_label else 'red'
    ax.set_title(
        f'True: {true_label}\nPred: {pred_label} ({conf:.0f}%)',
        color=color, fontsize=9
    )
    ax.axis('off')

plt.suptitle('Sample Predictions (Green=Correct, Red=Wrong)', fontsize=13)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Save the trained model
model.save('cifar10_cnn_model.h5')
print('Model saved to cifar10_cnn_model.h5')

## 9. Summary
| Component | Detail |
|-----------|--------|
| Dataset | CIFAR-10 (60,000 images, 32×32 RGB, 10 classes) |
| Architecture | 3× Conv blocks + Dense head |
| Regularization | BatchNorm + Dropout + Data Augmentation |
| Optimizer | Adam (lr=1e-3, ReduceLROnPlateau) |
| Early Stopping | Yes (patience=10, restore best weights) |
| Expected Test Accuracy | ~75–80% (without transfer learning) |

The CNN learns spatial hierarchies — edges → shapes → objects — through its stacked convolutional blocks, achieving solid accuracy on this 10-class benchmark.